In [124]:
from typing import TypedDict

from langchain.tools import tool
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph


In [125]:
# Define our single business tool
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an order that hasn't shipped."""
    # (Here you'd call your real backend API)
    return f"Order {order_id} has been cancelled."

In [126]:
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})

    prompt = f"""
    You are an ecommerce support agent.

    ORDER ID: {order['order_id']}

    If the customer asks to cancel,
    call cancel_order(order_id)
    and then send a simple confirmation.

    Otherwise, respond normally.
    """

    full = [SystemMessage(content=prompt)] + msgs

    llm = ChatOllama(
        model="qwen3:8b",
        temperature=0
    ).bind_tools([cancel_order])

    # First LLM call
    first = llm.invoke(full)

    out = [first]

    if first.tool_calls:
        tc = first.tool_calls[0]

        result = cancel_order.invoke(tc["args"])

        tool_message = ToolMessage(
            content=str(result),
            tool_call_id=tc["id"]
        )

        out.append(tool_message)

        # Second LLM call
        second = llm.invoke(full + out)

        out.append(second)

    return {
        "messages": out
    }

In [127]:
class AgentState(TypedDict):
    order: dict
    messages: list[BaseMessage]

In [128]:
# -- 3) Wire it all up in a StateGraph
def construct_graph():
    g = StateGraph(AgentState)
    g.add_node("assistant", call_model)
    g.set_entry_point("assistant")
    return g.compile()

In [129]:
graph = construct_graph()

example_order = {"order_id": "A12345"}
convo = [HumanMessage(content="Please cancel my order A12345.")]
result = graph.invoke({"order": example_order, "messages": convo})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}") 

ai: 
tool: Order A12345 has been cancelled.
ai: Your order A12345 has been successfully canceled. If you need any further assistance, feel free to ask!


In [133]:
example_order = {"order_id": "B73973"}

convo = [
    HumanMessage(
        content="Please cancel order #B73973. I found a cheaper option elsewhere."
    )
]

result = graph.invoke({
    "order": example_order,
    "messages": convo
})


# 1. Check that cancel_order was actually called
tool_called = any(
    hasattr(m, "tool_calls")
    and any(
        tc.get("name") == "cancel_order"
        for tc in m.tool_calls
    )
    for m in result["messages"]
)

assert tool_called, "Cancel order tool not called"


# 2. Check that cancellation was confirmed
confirmation_found = any(
    "cancelled" in str(m.content).lower()
    or "canceled" in str(m.content).lower()
    for m in result["messages"]
)

assert confirmation_found, "Confirmation message missing"


print("✅ Agent passed minimal evaluation.")

✅ Agent passed minimal evaluation.
